# Chapter 05-04 · Metrics: MAE, MSE, RMSE, MAPE's trouble, R-squared

**Label:** Core  |  **Time:** ~50 minutes  |  **Difficulty:** easy arithmetic, consequential choices

**Prerequisites:** 05-01 for the best constant per metric, 05-02 for R-squared as skill.

**Position in the learning path:** module 05, chapter 4 of 12.

---

## Why this matters

The last three chapters have quietly used five different ways of scoring a prediction. This one takes them
properly, because **the metric is the part of a modelling project that a stakeholder actually reads**, and
each of these five fails in a different way.

05-01 established the principle: a metric is a specification of what to predict, not merely a way to score.
This chapter is the practical follow-up - what each metric is *in the units of the problem*, which ones an
outlier can move, and the two that are routinely misread. One of them, MAPE, is probably the most-used
metric in business forecasting and it is broken in three separate ways.

## What you will be able to do

- State MAE, MSE, RMSE, MAPE and R-squared in the units of the problem
- Use the RMSE-to-MAE ratio as a free diagnostic of how uneven your errors are
- Predict how far each metric moves when one prediction goes badly wrong
- Explain MAPE's three failures, and say when to use it anyway
- Read a negative R-squared, and say what it means about the model

## Warm-up: retrieve, do not reread

1. Which constant minimises absolute error, and which minimises squared error?
2. What is R-squared, expressed as a skill score?
3. In 05-01, where did MAPE's best constant sit relative to the median?

<br>

*Answers: (1) the median and the mean. (2) skill under squared error against the mean baseline. (3) below
it - 31.0 against a median of 34.*

## Five numbers for one set of errors

The nine deliveries from 05-02 and the line fitted to them. Nine actuals, nine predictions, one set of
residuals - scored five ways.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# TINY: 05-02's nine deliveries and the least-squares line fitted to them
actual = np.array([16, 20, 25, 25, 30, 31, 36, 42, 45], dtype=float)
predicted = np.array([16, 19.5, 23, 26.5, 30, 33.5, 37, 40.5, 44], dtype=float)
error = actual - predicted


def mean_absolute(a, p):
    return float(np.abs(a - p).mean())


def mean_squared(a, p):
    return float(((a - p) ** 2).mean())


def root_mean_squared(a, p):
    return float(np.sqrt(mean_squared(a, p)))


def percentage(a, p):
    return float(np.mean(np.abs(a - p) / np.abs(a)))


def symmetric_percentage(a, p):
    return float(np.mean(2 * np.abs(a - p) / (np.abs(a) + np.abs(p))))


print("the nine residuals:", error)
print()
print("%-38s %10s   %s" % ("metric", "value", "units"))
print("%-38s %10.4f   %s" % ("mean absolute error (MAE)", mean_absolute(actual, predicted), "minutes"))
print("%-38s %10.4f   %s" % ("mean squared error (MSE)", mean_squared(actual, predicted),
                             "minutes SQUARED"))
print("%-38s %10.4f   %s" % ("root mean squared error (RMSE)", root_mean_squared(actual, predicted),
                             "minutes"))
print("%-38s %10.4f   %s" % ("mean absolute percentage error", percentage(actual, predicted),
                             "a fraction (3.68%)"))
print("%-38s %10.4f   %s" % ("symmetric MAPE", symmetric_percentage(actual, predicted),
                             "a fraction (3.67%)"))

**Only two of those five are in minutes.**

- **MAE = 1.1111 minutes.** The average size of a miss. If you tell a dispatcher one number, this is the
  one they can act on.
- **MSE = 1.8889 minutes squared.** Not interpretable as a quantity - nobody has an intuition for a
  squared minute. Its job is to be the thing that gets minimised, because it is smooth and has a closed
  form (05-02). **Report it almost never; optimise it often.**
- **RMSE = 1.3744 minutes.** MSE brought back into minutes by taking the square root, so it is
  comparable to MAE - and it is always at least as large.

That last fact is worth turning into a tool.

## The RMSE-to-MAE ratio is a free diagnostic

RMSE is never smaller than MAE, and **how much larger it is tells you how uneven your errors are.**

In [ ]:
profiles = [("every error the same", np.full(9, 2.0)),
            ("mildly varied", np.array([1, 1, 2, 2, 2, 2, 3, 3, 4], dtype=float)),
            ("one error carries it all", np.array([0, 0, 0, 0, 0, 0, 0, 0, 18], dtype=float))]

rows = []
for label, residuals in profiles:
    absolute = float(np.abs(residuals).mean())
    rooted = float(np.sqrt((residuals ** 2).mean()))
    rows.append({"error profile": label, "MAE": round(absolute, 3), "RMSE": round(rooted, 3),
                 "RMSE / MAE": round(rooted / absolute, 3)})
rows.append({"error profile": "the nine deliveries", "MAE": round(mean_absolute(actual, predicted), 3),
             "RMSE": round(root_mean_squared(actual, predicted), 3),
             "RMSE / MAE": round(root_mean_squared(actual, predicted) / mean_absolute(actual, predicted), 3)})
print(pd.DataFrame(rows).to_string(index=False))
print()
print("with 9 rows the ratio cannot exceed sqrt(9) = %.1f" % np.sqrt(9))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, (label, residuals) in zip(axes, profiles):
    absolute = float(np.abs(residuals).mean())
    rooted = float(np.sqrt((residuals ** 2).mean()))
    ax.bar(range(9), residuals, color="#0072B2", width=0.62)
    ax.axhline(absolute, color="#009E73", linewidth=2.2, label="MAE %.2f" % absolute)
    ax.axhline(rooted, color="#D55E00", linewidth=2.2, linestyle="--", label="RMSE %.2f" % rooted)
    ax.set_title("%s\nratio %.2f" % (label, rooted / absolute), fontsize=10.5)
    ax.set_xticks([])
    ax.legend(fontsize=8)
axes[0].set_ylabel("size of each error")
plt.tight_layout()
plt.show()

**Ratio 1.00 when every error is identical, 3.00 when one error is everything**, and the maximum possible
on nine rows is exactly 3 - the square root of the row count.

That gives a reading you get for nothing, from two numbers you were computing anyway:

| RMSE / MAE | Means |
|---|---|
| close to 1 | errors are all about the same size |
| around 1.2 to 1.4 | ordinary, roughly bell-shaped errors - the deliveries sit at 1.24 |
| above 2 | a small number of rows are producing most of the error |
| close to sqrt(n) | essentially one row is producing all of it |

**A high ratio is an instruction to go and look at the worst rows**, not to change metric. It is the
cheapest error analysis available, and 05-12 builds the full version.

## One prediction goes badly wrong

The last delivery took 45 minutes and the model now says 20.

**Predict before running:** by what factor does each metric get worse?

In [ ]:
broken = predicted.copy()
broken[-1] = 20.0

rows = []
for name, function in [("MAE", mean_absolute), ("MSE", mean_squared),
                       ("RMSE", root_mean_squared), ("MAPE", percentage)]:
    before, after = function(actual, predicted), function(actual, broken)
    rows.append({"metric": name, "before": round(before, 4), "after": round(after, 4),
                 "times worse": round(after / before, 2)})
damage = pd.DataFrame(rows)
print(damage.to_string(index=False))

fig, ax = plt.subplots(figsize=(8.5, 4.2))
bars = ax.bar(damage.metric, damage["times worse"],
              color=["#009E73", "#D55E00", "#e8a33d", "#7B3294"], width=0.6)
for bar, value in zip(bars, damage["times worse"]):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.9, "%.1fx" % value, ha="center",
            fontsize=11, fontweight="bold")
ax.set_ylabel("how many times worse the metric got")
ax.set_ylim(0, 43)
ax.set_title("One bad prediction in nine, seen by four metrics", fontsize=11.5)
plt.tight_layout()
plt.show()

**MSE gets 37.7 times worse. MAE gets 3.4 times worse.** RMSE sits between at 6.1, and MAPE is the least
disturbed at 2.6.

The ordering is not a curiosity, it is the definition of each metric showing through:

- **MAE** counts the error once, so one error 24 times the usual size adds roughly 24 usual errors.
- **MSE** squares it, so the same error counts about 600 times - and dominates everything.
- **RMSE** is the square root of that, which compresses it back but not all the way.
- **MAPE** divides by the actual, and this delivery has a large actual, so the miss is discounted.

**Which behaviour is right depends entirely on whether one three-hour delivery is genuinely 600 times
worse than a typical miss.** Sometimes it is - a missed surgical slot, a stockout of a critical part.
Usually it is not.

## MAPE, and why it keeps causing trouble

MAPE is the metric business audiences ask for, because a percentage sounds interpretable. It has **three
distinct problems**, and they are worth separating.

### Problem one: it is not symmetric

In [ ]:
ratios = np.linspace(0.05, 3.0, 300)
mape_curve = np.abs(1 - ratios)
smape_curve = 2 * np.abs(1 - ratios) / (1 + ratios)

fig, ax = plt.subplots(figsize=(9, 4.6))
ax.plot(ratios, 100 * mape_curve, color="#7B3294", linewidth=2.4, label="MAPE")
ax.plot(ratios, 100 * smape_curve, color="#0072B2", linewidth=2.4, linestyle="--", label="sMAPE")
ax.axvline(1.0, color="#666666", linewidth=1.2)
ax.axhline(100, color="#D55E00", linestyle=":", linewidth=1.6,
           label="under-prediction can never exceed 100%")
for ratio, offset, align in [(0.5, (12, 14), "left"), (2.0, (-12, 16), "right")]:
    ax.plot([ratio], [100 * abs(1 - ratio)], "o", color="#7B3294", markersize=9)
    ax.annotate("predict %.1fx\nMAPE %.0f%%" % (ratio, 100 * abs(1 - ratio)),
                (ratio, 100 * abs(1 - ratio)), textcoords="offset points", xytext=offset,
                fontsize=9, color="#7B3294", ha=align)
ax.set_xlabel("prediction divided by actual")
ax.set_ylabel("error (%)")
ax.set_ylim(0, 210)
ax.set_title("Predicting double is punished twice as hard as predicting half", fontsize=11.5)
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

for ratio in [0.25, 0.5, 1.0, 2.0, 4.0]:
    print("  predict %.2f x actual  ->  MAPE %6.1f%%   sMAPE %6.1f%%"
          % (ratio, 100 * abs(1 - ratio), 100 * 2 * abs(1 - ratio) / (1 + ratio)))

**Predicting double the truth scores 100%. Predicting half scores 50%.** Both are wrong by a factor of
two, in opposite directions, and MAPE calls one twice as bad as the other.

The consequence is structural: **under-prediction can never cost more than 100%, and over-prediction is
unbounded.** So any model or constant chosen to minimise MAPE will shade low - which is exactly what
05-01 measured, where MAPE's best constant was 31.0 against a median of 34.0 and a mean of 36.4.

**For a forecast that feeds ordering or staffing, that bias points the wrong way**: it systematically
under-orders, and the metric reports success while it does so.

`sMAPE` divides by the average of actual and predicted instead, which fixes the double-versus-half case -
both score 66.7% - at the price of a metric that is harder to explain and still misbehaves near zero.

### Problem two: it explodes near zero

In [ ]:
small_actuals = np.array([0.5, 2.0, 40.0])
small_predictions = np.array([1.0, 2.5, 40.5])

contributions = 100 * np.abs(small_actuals - small_predictions) / small_actuals
print(pd.DataFrame({"actual": small_actuals, "predicted": small_predictions,
                    "absolute error": np.abs(small_actuals - small_predictions),
                    "its percentage": np.round(contributions, 1)}).to_string(index=False))
print()
print("every absolute error is 0.5, and MAPE reports %.1f%%" % contributions.mean())
print("the first row alone contributes %.1f of those %.1f points"
      % (contributions[0] / 3, contributions.mean()))

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.1))

positions = np.arange(3)
labels = ["actual 0.5", "actual 2.0", "actual 40.0"]
left.bar(positions, np.abs(small_actuals - small_predictions), color="#0072B2", width=0.55)
left.set_xticks(positions)
left.set_xticklabels(labels, fontsize=9.5)
left.set_ylabel("absolute error")
left.set_ylim(0, 0.75)
left.set_title("The three errors are identical", fontsize=11)

right.bar(positions, contributions, color="#7B3294", width=0.55)
for position, value in zip(positions, contributions):
    right.text(position, value + 2.5, "%.1f%%" % value, ha="center", fontsize=10,
               fontweight="bold")
right.axhline(100 * percentage(small_actuals, small_predictions), color="#D55E00",
              linestyle="--", linewidth=2,
              label="MAPE = their mean, %.1f%%" % (100 * percentage(small_actuals, small_predictions)))
right.set_xticks(positions)
right.set_xticklabels(labels, fontsize=9.5)
right.set_ylabel("percentage error")
right.set_ylim(0, 118)
right.set_title("MAPE sees them as 100%, 25% and 1.2%", fontsize=11)
right.legend(fontsize=8.5)

plt.tight_layout()
plt.show()

**Three identical errors of 0.5, and MAPE calls the average error 42.1%** - because dividing 0.5 by an
actual of 0.5 gives 100%, while dividing it by 40 gives 1.2%.

**One row with a small actual can dominate the entire metric**, and if any actual is exactly zero MAPE is
undefined and most implementations return infinity or silently drop the row - which is worse, because the
number that comes back looks fine.

This is why MAPE is a poor choice for demand data with quiet periods, sensor data near a baseline, or
anything that can legitimately be zero.

### Problem three: it is not comparable across series

Because the denominator is the data itself, **a MAPE of 8% on high-volume items and 8% on low-volume items
do not describe the same quality of forecast**, and averaging MAPE across products silently weights the
small ones most.

### So when is MAPE reasonable?

When all three problems are absent: **actuals comfortably away from zero, on a similar scale, and no
strong asymmetry in what over- and under-prediction cost.** Prices, salaries, house values, journey times
all qualify. Demand data with intermittent zeros does not.

And if the audience wants a percentage, **weighted absolute percentage error** - total absolute error
divided by total actual - keeps the scale-free reading without letting a single small actual take over.

In [ ]:
def weighted_percentage(a, p):
    return float(np.abs(a - p).sum() / np.abs(a).sum())


print("on the three rows above:")
print("  MAPE                                : %.1f%%" % (100 * percentage(small_actuals, small_predictions)))
print("  weighted absolute percentage error  : %.1f%%"
      % (100 * weighted_percentage(small_actuals, small_predictions)))
print()
print("on the nine deliveries:")
print("  MAPE                                : %.2f%%" % (100 * percentage(actual, predicted)))
print("  weighted absolute percentage error  : %.2f%%" % (100 * weighted_percentage(actual, predicted)))

## R-squared, and the value people assume is impossible

05-02 established that R-squared **is** the skill score under squared error against the mean baseline. Two
consequences follow that surprise people, and both matter.

**It can be negative.** Skill is negative whenever the model is worse than the baseline, and on data the
model has not seen that happens easily.

**On the fitting rows it cannot go down when you add a feature** - 05-03's E9 watched it climb to 0.85 on
pure noise columns. So R-squared measured where the model was fitted is not evidence of anything.

Here are both, in one experiment: a polynomial fitted to 25 points, at rising degrees.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import PolynomialFeatures

fit_rng = np.random.default_rng(2)
fit_x = fit_rng.uniform(-3, 3, 25)
fit_y = 0.8 * fit_x + fit_rng.normal(0, 1.0, 25)
fresh_x = fit_rng.uniform(-3, 3, 200)
fresh_y = 0.8 * fresh_x + fit_rng.normal(0, 1.0, 200)

rows = []
for degree in [1, 3, 7, 12, 18]:
    expand = PolynomialFeatures(degree, include_bias=False)
    fitted = LinearRegression().fit(expand.fit_transform(fit_x.reshape(-1, 1)), fit_y)
    rows.append({"polynomial degree": degree,
                 "R2 on the 25 fitting rows": round(float(r2_score(
                     fit_y, fitted.predict(expand.transform(fit_x.reshape(-1, 1))))), 4),
                 "R2 on 200 fresh rows": round(float(r2_score(
                     fresh_y, fitted.predict(expand.transform(fresh_x.reshape(-1, 1))))), 4)})
degrees = pd.DataFrame(rows)
print(degrees.to_string(index=False))

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.4))

left.plot(degrees["polynomial degree"], degrees["R2 on the 25 fitting rows"], "o-",
          color="#0072B2", linewidth=2.2, markersize=8, label="on the fitting rows")
left.plot(degrees["polynomial degree"], degrees["R2 on 200 fresh rows"], "s-",
          color="#D55E00", linewidth=2.2, markersize=8, label="on fresh rows")
left.axhline(0, color="#000000", linewidth=1.4)
left.set_ylim(-2, 1.05)
left.set_xlabel("polynomial degree")
left.set_ylabel("R-squared")
left.set_title("Zoomed to the readable range", fontsize=11)
left.legend(fontsize=8.5)

right.plot(degrees["polynomial degree"], degrees["R2 on the 25 fitting rows"], "o-",
           color="#0072B2", linewidth=2.2, markersize=8)
right.plot(degrees["polynomial degree"], degrees["R2 on 200 fresh rows"], "s-",
           color="#D55E00", linewidth=2.2, markersize=8)
right.axhline(0, color="#000000", linewidth=1.4)
right.set_yscale("symlog", linthresh=1)
right.set_xlabel("polynomial degree")
right.set_ylabel("R-squared (symmetric log scale)")
right.set_title("The full range: fresh-row R2 reaches %.0f" % degrees["R2 on 200 fresh rows"].min(),
                fontsize=11)

plt.tight_layout()
plt.show()

**On the fitting rows R-squared climbs to about 0.83. On fresh rows it goes to -1850.87.**

One detail in that table is worth stopping on, because it contradicts what was said two cells ago.
Degree 12 scores 0.8337 on the fitting rows and degree 18 scores 0.8048 - it went **down**, even though
degree 18 contains every column degree 12 has. In exact arithmetic that is impossible.

What broke is not the theory but the arithmetic. On this data `x` reaches 3, so the degree-18 column
reaches 3^18, about 387 million, while the first column is around 3. Columns on wildly different scales
make the least-squares solve numerically unstable, and the solver returns something near the optimum
rather than the optimum. **Take this as a small preview of a real failure mode:** raw powers are a
badly-behaved way to build polynomial features, and 05-07 will return to what to do instead.

A negative R-squared has a precise reading, and it is not "the model explains no variance":

> **R-squared of -1850 means the model's squared error is 1,851 times larger than simply predicting the
> mean.** It is not merely useless, it is actively worse than a constant - which is what 04-02 called
> negative skill.

The two curves also make the reporting rule concrete: **R-squared quoted without saying which rows it was
measured on is uninterpretable.** The same model here can be described as 0.80 or as -1850 truthfully.

### Adjusted R-squared, and why it is not the answer

Because R-squared cannot fall when a feature is added, a variant exists that penalises the count:
`1 - (1 - R²)(n - 1)/(n - p - 1)`.

In [ ]:
def adjusted(r_squared, rows_used, features_used):
    return 1 - (1 - r_squared) * (rows_used - 1) / (rows_used - features_used - 1)


noise_rng = np.random.default_rng(11)
design = fit_x.reshape(-1, 1)
rows = []
for extra in [0, 5, 10, 15, 20]:
    columns = np.column_stack([design] + [noise_rng.normal(size=25) for _ in range(extra)])
    fitted = LinearRegression().fit(columns, fit_y)
    plain = float(fitted.score(columns, fit_y))
    rows.append({"noise columns added": extra, "features": columns.shape[1],
                 "R2": round(plain, 4),
                 "adjusted R2": round(adjusted(plain, 25, columns.shape[1]), 4)})
print(pd.DataFrame(rows).to_string(index=False))

**Adjusted R-squared does push back.** Plain R-squared climbs from 0.52 to 0.91 as twenty pure-noise
columns are added; the adjusted version falls from 0.50 to 0.18 over most of that range.

But look at the final row, where it rises again to 0.27. With 21 features on 25 rows the denominator
`n - p - 1` is **3**, and the correction becomes wild - it is dividing by a number about to hit zero.
**The formula degrades exactly where you would most want it to work**, which is the honest limit of a
correction that counts parameters instead of holding data out.

**It is a correction, not a substitute for held-out data.** The honest version of the same question is the
right-hand curve above: fit on some rows, score on others. Adjusted R-squared is useful when you cannot
hold data out at all, and 04-03 already established what to do when you can.

## Choosing, and reporting

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.4))
guide = [
    ("MAE", "#cfe8dc", "minutes", "low", "none worth naming",
     "the default to report - what a stakeholder can act on"),
    ("MSE", "#cfe3f3", "minutes squared", "very high", "not interpretable",
     "optimise it, do not report it"),
    ("RMSE", "#cfe3f3", "minutes", "high", "none",
     "report alongside MAE; the ratio is a free diagnostic"),
    ("MAPE", "#f3d6e3", "percent", "low", "asymmetric, explodes near 0",
     "only with actuals far from zero and on one scale"),
    ("R-squared", "#f6d3bd", "unitless", "high", "rises with any feature",
     "say which rows it was measured on, or do not quote it"),
]
for position, (name, colour, units, sensitivity, failure, advice) in enumerate(guide):
    y = len(guide) - position - 1
    ax.add_patch(plt.Rectangle((0.05, y + 0.08), 1.5, 0.84, facecolor=colour, edgecolor="white",
                               linewidth=2.5))
    ax.text(0.8, y + 0.5, name, ha="center", va="center", fontsize=12, fontweight="bold")
    ax.text(1.75, y + 0.66, "units: %s   |   outlier sensitivity: %s" % (units, sensitivity),
            va="center", fontsize=9, color="#555555")
    ax.text(1.75, y + 0.34, "%s  -  %s" % (failure, advice), va="center", fontsize=9.5,
            color="#222222")
ax.set_xlim(0, 11.6)
ax.set_ylim(-0.15, len(guide) + 0.35)
ax.set_xticks([]); ax.set_yticks([])
for side in ax.spines.values():
    side.set_visible(False)
ax.set_title("Five metrics, and what each is for", fontsize=13)
plt.tight_layout()
plt.show()

**What to actually report**, and it is short:

> **MAE in the units of the problem, the baseline it beats, and the spread across splits.** Add RMSE if
> large errors matter more, and say so. Add R-squared only with the rows named.

That sentence carries 05-01's principle (the metric is a choice), 04-02's (a number without a baseline is
not a result) and 04-03's (a single split is one draw). The rest of this module will report exactly that.

## Common misconceptions

**"RMSE and MAE measure the same thing in the same units."**
Same units, different quantities. RMSE is always at least MAE, and the gap is information about how uneven
the errors are.

**"MSE is a good thing to report."**
It is measured in squared units. Optimise it; report its square root.

**"MAPE is interpretable because it is a percentage."**
It is asymmetric, undefined at zero, and dominated by small actuals. It is interpretable only when none of
those apply.

**"sMAPE fixes MAPE."**
It fixes the asymmetry between doubling and halving. It is still unstable near zero and harder to explain.

**"A high R-squared means a good model."**
On the fitting rows it means you added features. The number that matters is on rows the model has not seen,
and there it can be negative.

**"A negative R-squared means something went wrong."**
It means the model is worse than predicting the mean. That is a result, and often a correct one.

**"Pick the metric that makes the model look best."**
The metric determines what the best model *is*. Picking it afterwards means you optimised for one thing
and reported another.

## Exercises

Solutions: `solutions/05_regression/05-04_metrics_solutions.ipynb`.

### Quick understanding

**E1.** Give the units of MAE, MSE, RMSE, MAPE and R-squared for a model predicting minutes.

**E2.** Why is RMSE never smaller than MAE, and what does a ratio of 1 tell you?

**E3.** Name MAPE's three failures in one clause each.

### Hand calculation

**E4.** Errors of 1, 1, 1, 5. Compute MAE, MSE and RMSE by hand, and the RMSE-to-MAE ratio.

**E5.** For the same four errors, remove the 5 and replace it with 1. Recompute all three. Which metric
changed proportionally the most, and by what factor?

**E6.** Actual 80, predicted 100. Compute MAPE and sMAPE. Then actual 100, predicted 80. Compute both
again. Which metric treats the two cases the same, and which does not?

**E7.** A model has MSE 400 on a target measured in euros. Give RMSE, and say which of the two you would
put in a report and why.

### Coding

**E8.** Write `report(actual, predicted, baseline)` printing MAE, RMSE, the RMSE-to-MAE ratio, and the
skill against the baseline. Run it on the nine deliveries against the median constant.

**E9.** Simulate 2,000 datasets of 50 errors drawn from a normal distribution and record the RMSE-to-MAE
ratio each time. What value does it cluster around? Repeat with errors from a heavy-tailed distribution
and report the difference.

**E10.** Find the constant that minimises MAPE on the nine delivery *actuals* by grid search, and compare
it with the median and the mean. Confirm 05-01's finding, then explain the ordering from the asymmetry
curve.

**E11.** Take a model that is systematically 10% high and one that is systematically 10% low. Score both
with MAPE, sMAPE and MAE. Which metrics can tell them apart, and which cannot?

**E12.** Write `r2_by_rows(model, X_fit, y_fit, X_fresh, y_fresh)` reporting R-squared on both sets, and
use it to find the polynomial degree at which the fresh-row R-squared first goes negative on this
chapter's data.

### Interpretation

**E13.** A forecasting team reports "MAPE 12%, down from 15% last quarter" and wants to celebrate. Give
two ways that could happen without the forecasts improving at all.

**E14.** A model reports RMSE 4.2 and MAE 1.1 on 500 rows. What do you conclude, and what is the first
thing you look at?

### Debugging

**E15.** Your MAPE is `inf`. Name the cause and two things you could do, and say which one you would
actually choose.

**E16.** A colleague's R-squared is 0.92 on training rows and 0.10 on test rows, and they conclude the test
set is "unrepresentative". Give the more likely explanation and the check.

### Exam and interview reasoning

**E17.** "Which metric would you use for a regression problem?" Answer in under a minute, then handle:
"the business always asks for MAPE - what do you tell them?"

### Transfer to a different situation

**E18.** You are forecasting daily electricity demand, where demand is never near zero, occasional extreme
days matter enormously, and under-forecasting causes blackouts while over-forecasting wastes money. State
the metric you would optimise, the metric you would report, and why they differ.

### Explain it to someone non-technical

**E19.** Explain in under 90 words why "our forecast is 95% accurate" is not a meaningful statement without
more detail.

### Optional challenge

**E20.** Show that the RMSE-to-MAE ratio is bounded between 1 and `sqrt(n)`, by finding the error vectors
that achieve each bound for `n = 9`. Then show empirically what the ratio converges to for normally
distributed errors as `n` grows, and identify the constant.

In [ ]:
# Your workspace. In memory: actual, predicted, error, mean_absolute, mean_squared,
# root_mean_squared, percentage, symmetric_percentage, weighted_percentage, broken,
# profiles, degrees, adjusted.

## Mastery check

- [ ] State every metric's units without looking
- [ ] Use the RMSE-to-MAE ratio to decide whether to go and look at individual rows
- [ ] Predict which metric an outlier will move most, and roughly by how much
- [ ] Explain MAPE's asymmetry and why it biases forecasts low
- [ ] Read a negative R-squared as "worse than the mean, by this factor"
- [ ] Report a result in the form the rest of this module will use

## What should now feel instinctive

- Quoting an error in the units of the problem, with the baseline attached
- Computing RMSE and MAE together and glancing at the ratio
- Refusing MAPE on data that can be near zero
- Asking "measured on which rows?" of any R-squared
- Separating the metric you optimise from the metric you report, deliberately

## Flashcards

| Front | Back |
|---|---|
| MAE | Average size of a miss, in the target's units. The default to report |
| MSE | In squared units. Optimise it, never report it |
| RMSE | MSE brought back into the target's units. Always at least MAE |
| RMSE / MAE | 1 when errors are equal, sqrt(n) when one row carries everything. 1.24 here |
| One bad prediction in nine | MSE 37.7x worse, RMSE 6.1x, MAE 3.4x, MAPE 2.6x |
| MAPE's asymmetry | Predicting double costs 100%, predicting half costs 50% |
| Why MAPE biases low | Under-prediction is capped at 100%, over-prediction is unbounded |
| MAPE near zero | Three equal errors of 0.5 gave 42.1%, all of it from one small actual |
| A safer percentage | Weighted absolute percentage error: total error over total actual |
| R-squared | Skill under squared error against the mean. Negative means worse than a constant |
| R-squared of -1850 | The squared error is 1,851 times the mean's |
| What to report | MAE, its baseline, and the spread across splits |

## Next

**05-05 · Residuals: reading the errors the model leaves.** Every metric in this chapter collapses the
residuals into a single number, and that is what makes them reportable - and what makes them blind. The
next chapter puts the residuals back on a plot, where a model with an excellent RMSE can be seen to be
wrong in a specific, fixable way: curvature it missed, variance that grows with the prediction, and rows
it fails on systematically.

Two models with identical MAE can leave completely different residuals, and only one of them is finished.